# pandas Zero to Hero

A detailed, ground-up pandas refresher: Series/DataFrame basics through groupby, merging, and
performance — with the project's own `window_metadata.parquet` as the running real-data example.
Companion notebooks: `refresher_numpy.ipynb`, `refresher_pytorch.ipynb`, and
`refresher_deep_learning.ipynb` (concepts).

**Contents**
1. Why pandas (labeled data on top of NumPy)
2. Series
3. DataFrame: creation, dtypes, inspection
4. Reading and writing files
5. Indexing: `[]`, `.loc`, `.iloc`, `.at`/`.iat`
6. Filtering rows with boolean conditions
7. Missing data
8. Adding and modifying columns
9. Sorting
10. GroupBy, in depth
11. Merging, joining, concatenating
12. Reshaping: pivot tables, melt, stack/unstack
13. String methods
14. Datetime handling
15. MultiIndex, briefly
16. Performance: vectorization over `.apply`/`.iterrows`
17. Worked example: `window_metadata.parquet`, end to end
18. Common pitfalls, summarized

Run this notebook from the repo root (CLAUDE.md rule 5) — §4 and §17 read
`data/prepared_data/window_metadata.parquet` relative to the working directory.


## 1. Why pandas

NumPy arrays are homogeneous and positional (`arr[3, 1]`, no names). Real datasets are usually
*heterogeneous* (a mix of strings, numbers, dates) and *labeled* (columns have names, rows have
meaningful identifiers). pandas adds exactly that layer on top of NumPy: labeled, heterogeneous,
2D (`DataFrame`) or 1D (`Series`) tables, with fast label-aware operations.

Every window in `window_metadata.parquet` is a row with named columns — `sender_uid`, `family`,
`attack_label`, `window_start_idx`, ... — exactly the kind of table pandas exists for.


## 2. Series

A `Series` is one labeled 1D array: values (a NumPy array under the hood) plus an **index**
(labels for each value). Think of it as a single column that remembers its own row labels.


In [ ]:
import pandas as pd
import numpy as np

s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])
print(s)
print("\nvalues (a numpy array):", s.values, type(s.values))
print("index:", s.index.tolist())
print("s['b']:", s["b"])           # label-based access
print("s.iloc[1]:", s.iloc[1])     # position-based access, same value

# Series operations are vectorized and ALIGN by index, not position -- a key pandas idea.
s2 = pd.Series([1, 1, 1, 1], index=["b", "c", "d", "e"])
print("\ns + s2 (aligned by label, mismatches become NaN):")
print(s + s2)


## 3. DataFrame: creation, dtypes, inspection

A `DataFrame` is a dict of `Series` sharing one index — conceptually, a table where every column
can have its own dtype.


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "sender_uid": ["a1", "a2", "a3", "a4"],
    "attack_label": ["Benign", "GridSybil", "GridSybil", "DataReplaySybil"],
    "n_windows": [12, 340, 7, 88],
    "is_attacker": [False, True, True, True],
})
print(df)
print("\ndf.shape:", df.shape)
print("df.dtypes:\n", df.dtypes)
print("\ndf.columns:", df.columns.tolist())
print("df.index:", df.index.tolist())

print("\ndf.info():")
df.info()

print("\ndf.describe() -- summary stats, numeric columns only by default:")
print(df.describe())


In [ ]:
# Selecting a column returns a Series; a list of columns returns a DataFrame.
print(type(df["attack_label"]).__name__, "for df['attack_label']")
print(type(df[["attack_label", "n_windows"]]).__name__, "for df[['attack_label', 'n_windows']]")

print("\nsingle column:\n", df["n_windows"])
print("\nmultiple columns:\n", df[["sender_uid", "n_windows"]])


## 4. Reading and writing files

pandas reads/writes most common formats with one function call each: `read_csv`, `read_parquet`,
`read_json`, and their `to_*` counterparts. This project stores its authoritative window metadata
as **parquet** (`window_metadata.parquet`) — columnar, compressed, and dtype-preserving, unlike CSV
which round-trips everything through text.


In [ ]:
import pandas as pd
import tempfile, pathlib

df = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"], "c": [1.5, 2.5, 3.5]})

with tempfile.TemporaryDirectory() as tmp:
    tmp = pathlib.Path(tmp)

    df.to_csv(tmp / "df.csv", index=False)
    from_csv = pd.read_csv(tmp / "df.csv")
    print("csv round-trip dtypes:\n", from_csv.dtypes)   # ints/floats inferred back correctly

    df.to_parquet(tmp / "df.parquet")
    from_parquet = pd.read_parquet(tmp / "df.parquet")
    print("\nparquet round-trip dtypes:\n", from_parquet.dtypes)

    df.to_json(tmp / "df.json", orient="records")
    print("\nfirst 60 chars of the json:", (tmp / "df.json").read_text()[:60])


## 5. Indexing: `[]`, `.loc`, `.iloc`, `.at`/`.iat`

Four ways to index a DataFrame, each with a distinct job. Mixing them up is the most common source
of pandas confusion.

| Accessor | Selects by | Typical use |
|---|---|---|
| `df[...]` | column name(s), or a boolean mask | quick column access, filtering |
| `df.loc[...]` | **label** (row label, column name) | the general-purpose label accessor |
| `df.iloc[...]` | **integer position** | when you need "row 3", regardless of its label |
| `df.at[...]` / `.iat[...]` | a single label / position | fastest way to get/set *one* scalar |


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "family": ["Benign", "GridSybil", "GridSybil", "DoSRandomSybil"],
    "n_windows": [12, 340, 7, 55],
}, index=["u0", "u1", "u2", "u3"])

print("df.loc['u1']:\n", df.loc["u1"])                      # row by label
print("\ndf.loc['u1', 'n_windows']:", df.loc["u1", "n_windows"])   # (row label, col label)
print("df.loc[['u0', 'u2'], 'family']:\n", df.loc[["u0", "u2"], "family"])

print("\ndf.iloc[1]:\n", df.iloc[1])                        # row by position (same row as 'u1')
print("df.iloc[0:2, 1]:\n", df.iloc[0:2, 1])                 # position slice, both axes

print("\ndf.at['u1', 'n_windows']:", df.at["u1", "n_windows"])   # fast scalar get
df.at["u1", "n_windows"] = 999                                     # fast scalar set, in place
print("after df.at assignment:\n", df)


## 6. Filtering rows with boolean conditions

Boolean masks work exactly like NumPy's (§6 of the NumPy notebook), applied to a whole DataFrame.
Combine conditions with `&`/`|`/`~` (not Python's `and`/`or`/`not`) and **parenthesize each
condition** — operator precedence makes `df.a > 1 & df.b < 2` a bug without the parens.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "family": ["Benign", "GridSybil", "GridSybil", "DoSRandomSybil", "Benign"],
    "group": ["0709", "0709", "1416", "1416", "1416"],
    "n_windows": [12, 340, 7, 55, 9],
})

attackers = df[df["family"] != "Benign"]
print("attackers only:\n", attackers)

both_conditions = df[(df["family"] == "GridSybil") & (df["group"] == "0709")]
print("\nGridSybil in 0709 (note the parens around each condition):\n", both_conditions)

either = df[(df["family"] == "Benign") | (df["n_windows"] > 100)]
print("\nbenign OR many windows:\n", either)

few_families = df[df["family"].isin(["Benign", "GridSybil"])]
print("\nisin() -- membership test, cleaner than chained ==:\n", few_families)

# query() -- the same filter as a string expression; handy for long conditions.
print("\nsame result via .query():\n", df.query("family == 'GridSybil' and group == '0709'"))


## 7. Missing data

pandas represents "no value" as `NaN` (float) or `pd.NA` (nullable dtypes) or `NaT` (missing
datetimes). Most aggregations (`.mean()`, `.sum()`) **skip NaN by default** — different from NumPy,
where a single `NaN` poisons the whole reduction unless you use the `np.nan*` variants.


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({"a": [1.0, np.nan, 3.0, np.nan], "b": [10.0, 20.0, np.nan, 40.0]})
print(df)

print("\nisna():\n", df.isna())
print("\nrows with ANY missing value dropped:\n", df.dropna())
print("\nrows with ALL values missing dropped (none here):\n", df.dropna(how="all"))

print("\nfillna(0):\n", df.fillna(0))
print("\nfillna with per-column value:\n", df.fillna({"a": df["a"].mean(), "b": 0}))

print("\nmean() skips NaN by default:", df["a"].mean())          # (1+3)/2 = 2.0, not NaN
print("sum() treats NaN as 0 by default:", df["a"].sum())          # 4.0


## 8. Adding and modifying columns

New columns are usually **vectorized expressions** over existing ones — the pandas equivalent of
NumPy broadcasting, and just as important to reach for instead of a per-row loop (see §16).


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({"spd_x": [3.0, 0.0, 5.0], "spd_y": [4.0, 2.0, 12.0], "attack_label": ["Benign", "GridSybil", "GridSybil"]})

df["speed"] = np.hypot(df["spd_x"], df["spd_y"])              # vectorized, elementwise
df["is_attacker"] = df["attack_label"] != "Benign"             # vectorized boolean
print(df)

# .map() -- element-by-element substitution via a dict or function (Series only).
severity = {"Benign": 0, "GridSybil": 3}
df["severity"] = df["attack_label"].map(severity)
print("\nafter .map():\n", df)

# .assign() -- returns a NEW DataFrame with added columns, doesn't mutate in place; chains well.
df2 = df.assign(speed_kmh=lambda d: d["speed"] * 3.6)
print("\n.assign() result has speed_kmh; original df does not:", "speed_kmh" in df.columns)


## 9. Sorting


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "sender_uid": ["u0", "u1", "u2", "u3"],
    "family": ["GridSybil", "Benign", "GridSybil", "DataReplaySybil"],
    "n_windows": [340, 12, 7589, 88],
})

print("sorted by n_windows descending:\n", df.sort_values("n_windows", ascending=False))
print("\nsorted by family then n_windows:\n", df.sort_values(["family", "n_windows"]))
print("\ntop 2 by n_windows (nlargest, faster than sort_values().head()):\n", df.nlargest(2, "n_windows"))
print("\nsorted by index:\n", df.sort_index())


## 10. GroupBy, in depth

**Split → apply → combine**: split rows into groups sharing a key, apply a function to each
group independently, combine the results back into one object. This is the single most-used
pandas operation in this project — it's exactly how `WindowStore.identity_table()`
(`scripts/export_simulation_sample.py`) turns 285,926 window rows into one row per identity, and
it's exactly the mechanism behind the GridSybil_0709 windowing defect (grouping by `senderPseudo`
*alone* silently merged 653 distinct physical vehicles into one group — see
`docs/research-notes/data_understanding/gridsybil-windowing-defect.md`).


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "family": ["Benign", "GridSybil", "GridSybil", "GridSybil", "DoSRandomSybil"],
    "group": ["0709", "0709", "0709", "1416", "1416"],
    "n_windows": [12, 340, 7589, 51, 22],
})

grouped = df.groupby("family")
print("number of groups:", grouped.ngroups)
print("group sizes:\n", grouped.size())

print("\n.sum() per group:\n", grouped["n_windows"].sum())
print("\n.agg() with multiple functions at once:\n", grouped["n_windows"].agg(["sum", "mean", "max", "count"]))

# Grouping by MULTIPLE columns -- each unique combination is one group.
by_family_group = df.groupby(["family", "group"])["n_windows"].sum()
print("\ngrouped by (family, group):\n", by_family_group)

# .transform() broadcasts a per-group result back to the ORIGINAL shape (unlike .agg(), which
# collapses). Useful for "what fraction of my group's total am I?" style columns.
df["group_total"] = df.groupby("family")["n_windows"].transform("sum")
print("\nper-row group_total via transform():\n", df)

# .get_group() pulls out one specific group as its own DataFrame -- this is exactly how
# WindowStore.rows_for(uid) finds one identity's windows.
print("\nget_group('GridSybil'):\n", grouped.get_group("GridSybil"))

# Named aggregation -- clean column names for multiple different aggregations at once.
summary = df.groupby("family").agg(
    total_windows=("n_windows", "sum"),
    n_pseudonyms=("n_windows", "count"),
)
print("\nnamed aggregation:\n", summary)


## 11. Merging, joining, concatenating

Three ways to combine DataFrames: `concat` stacks them (rows or columns); `merge` joins on shared
column values (SQL-style); `.join()` is `merge` specialized for joining on the *index*.


In [ ]:
import pandas as pd

top = pd.DataFrame({"x": [1, 2]})
bottom = pd.DataFrame({"x": [3, 4]})
print("concat rows (axis=0, default):\n", pd.concat([top, bottom], ignore_index=True))

left_cols = pd.DataFrame({"a": [1, 2]})
right_cols = pd.DataFrame({"b": [3, 4]})
print("\nconcat columns (axis=1):\n", pd.concat([left_cols, right_cols], axis=1))

identities = pd.DataFrame({"sender_uid": ["u0", "u1", "u2"], "family": ["Benign", "GridSybil", "GridSybil"]})
senders = pd.DataFrame({"sender_uid": ["u0", "u1", "u3"], "sender": [100, 200, 300]})

print("\ninner merge (only rows present in BOTH):\n", identities.merge(senders, on="sender_uid", how="inner"))
print("\nleft merge (all of `identities`, NaN where `senders` has no match):\n",
      identities.merge(senders, on="sender_uid", how="left"))
print("\nouter merge (union of both, NaN on either side):\n", identities.merge(senders, on="sender_uid", how="outer"))

# .join() is merge-on-index -- handy when the key is already the index.
left_idx = identities.set_index("sender_uid")
right_idx = senders.set_index("sender_uid")
print("\n.join() (merge on index):\n", left_idx.join(right_idx, how="left"))


## 12. Reshaping: pivot tables, melt, stack/unstack

**`pivot_table`**: turn one column's *values* into new columns (wide format) while aggregating.
**`melt`**: the inverse — turn columns *into* rows (wide → long, "tidy data").


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "family": ["Benign", "Benign", "GridSybil", "GridSybil"],
    "group": ["0709", "1416", "0709", "1416"],
    "n_windows": [12, 15, 340, 51],
})

wide = df.pivot_table(index="family", columns="group", values="n_windows", aggfunc="sum")
print("pivoted wide (family x group):\n", wide)

long_again = wide.reset_index().melt(id_vars="family", value_name="n_windows")
print("\nmelted back to long/tidy format:\n", long_again)

# stack()/unstack() move a column level into/out of the (row) index -- pivot_table's lower-level
# building blocks. Rarely needed directly once you know pivot_table/melt, but good to recognize.
print("\nwide.stack() (columns -> innermost index level):\n", wide.stack())


## 13. String methods

The `.str` accessor applies Python string methods elementwise across a Series, vectorized — no
`.apply(lambda s: s.upper())` needed.


In [ ]:
import pandas as pd

uids = pd.Series([
    "GridSybil_0709_VeReMi_25200_28800_2022-9-13_21:7:46_1083792",
    "Benign_1416_VeReMi_50400_54000_2022-9-11_19:13:55_1059315",
])

print("upper:\n", uids.str.upper().str[:20])
print("\ncontains 'Grid':\n", uids.str.contains("Grid"))
print("\nsplit on '_', first field (the family):\n", uids.str.split("_").str[0])
print("\nlength:\n", uids.str.len())
print("\nreplace:\n", uids.str.replace("VeReMi", "run", regex=False).str[:30])


## 14. Datetime handling

`pd.to_datetime` parses strings/numbers into real datetime objects; the `.dt` accessor then
extracts components, same spirit as `.str` for strings.


In [ ]:
import pandas as pd

raw = pd.Series(["2026-09-17 08:00:00", "2026-09-17 08:00:10", "2026-09-17 08:05:00"])
ts = pd.to_datetime(raw)
print(ts)
print("\n.dt.hour:", ts.dt.hour.tolist())
print(".dt.second:", ts.dt.second.tolist())

# Differences between timestamps are Timedeltas -- directly comparable/summable, like the
# project's own `dt` (seconds between consecutive BSMs).
gaps = ts.diff()
print("\nconsecutive gaps:\n", gaps)
print("gaps in seconds:", gaps.dt.total_seconds().tolist())


## 15. MultiIndex, briefly

Grouping by more than one column (§10) naturally produces a **MultiIndex** — a row index with
more than one level. `.xs()` and tuple indexing select from it; `.reset_index()` flattens it back
to plain columns, which is usually the simplest way to keep working with it.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "family": ["Benign", "GridSybil", "GridSybil", "DoSRandomSybil"],
    "group": ["0709", "0709", "1416", "1416"],
    "n_windows": [12, 340, 51, 22],
})
grouped = df.groupby(["family", "group"])["n_windows"].sum()
print("MultiIndex result:\n", grouped)
print("\ntype of index:", type(grouped.index).__name__)

print("\ntuple indexing, one specific (family, group):", grouped[("GridSybil", "0709")])
print("\n.xs() selects across one level, fixing the other:\n", grouped.xs("GridSybil", level="family"))
print("\n.reset_index() flattens back to plain columns:\n", grouped.reset_index())


## 16. Performance: vectorization over `.apply`/`.iterrows`

The pandas version of the NumPy notebook's §15: `.iterrows()` and `.apply()` with a Python
function both drop back into row-by-row Python — slow, and usually avoidable with a vectorized
column expression (§8) or a built-in `.str`/`.dt`/groupby method.


In [ ]:
import pandas as pd
import numpy as np
import time

n = 200_000
df = pd.DataFrame({"spd_x": np.random.default_rng(0).normal(size=n),
                    "spd_y": np.random.default_rng(1).normal(size=n)})

t0 = time.perf_counter()
speed_iterrows = [np.hypot(row.spd_x, row.spd_y) for _, row in df.iterrows()]
t_iterrows = time.perf_counter() - t0

t0 = time.perf_counter()
speed_apply = df.apply(lambda row: np.hypot(row["spd_x"], row["spd_y"]), axis=1)
t_apply = time.perf_counter() - t0

t0 = time.perf_counter()
speed_vectorized = np.hypot(df["spd_x"], df["spd_y"])
t_vectorized = time.perf_counter() - t0

print(f"iterrows:   {t_iterrows*1000:.0f} ms")
print(f"apply:      {t_apply*1000:.0f} ms")
print(f"vectorized: {t_vectorized*1000:.1f} ms  ({t_iterrows/t_vectorized:.0f}x faster than iterrows)")
assert np.allclose(speed_iterrows, speed_vectorized)


## 17. Worked example: `window_metadata.parquet`, end to end

Every technique above, applied to the real, committed dataset — read-only (CLAUDE.md rule 2).


In [ ]:
import pathlib
import pandas as pd

meta_path = pathlib.Path("data/prepared_data/window_metadata.parquet")
assert meta_path.exists(), "run this notebook from the repo root (CLAUDE.md rule 5) -- data/ not found here"

meta = pd.read_parquet(meta_path)
print(meta.shape, "rows x cols")
print(meta.dtypes)
meta[["sender_uid", "family", "group", "attack_label", "window_start_idx"]].head()


In [ ]:
# One row per physical identity with its window count -- exactly WindowStore.identity_table()
# in scripts/export_simulation_sample.py.
by_uid = meta.groupby("sender_uid").size().rename("n_windows").sort_values(ascending=False)
print("largest sequence groups (top 5) -- the GridSybil_0709 splice shows up right here:")
print(by_uid.head())

print("\nwindows per attack_label (class balance):")
print(meta.groupby("attack_label").size())

print("\nwindows per (attack_label, group) -- pivoted wide, like docs/research-notes/data_understanding/class-balance.md:")
print(meta.pivot_table(index="attack_label", columns="group", values="window_start_idx", aggfunc="count", fill_value=0))


In [ ]:
# Filtering + boolean indexing on real data: isolate exactly the known-defective splice group.
splice_mask = (meta["family"] == "GridSybil") & (meta["group"] == "0709") & (meta["senderPseudo"] == 1)
splice_rows = meta[splice_mask]
print("rows in the GridSybil_0709 senderPseudo==1 splice:", len(splice_rows))
print("as a fraction of the full corpus:", round(len(splice_rows) / len(meta) * 100, 2), "%")

# .nunique() -- how many distinct physical senders got merged under one sentinel pseudonym. This
# table doesn't carry the raw  column (that's resolved separately, from the ground-truth
# ledgers, in scripts/prepare_data.py's GroundTruthLoader) -- named here to show the pattern even
# though it can't run against this specific file.
if "sender" in meta.columns:
    print("distinct physical senders sharing that pseudonym:", meta.loc[splice_mask, "sender"].nunique())
else:
    print("'sender' column not in window_metadata.parquet -- see prepare_data.py's GroundTruthLoader")


## 18. Common pitfalls, summarized

- **`SettingWithCopyWarning`** — assigning into a slice of a DataFrame that pandas isn't sure is a
  view or a copy (`df[df.a > 0]["b"] = 1`, chained indexing). Fix: use `.loc` in one step
  (`df.loc[df.a > 0, "b"] = 1`), or `.copy()` explicitly when you mean to detach a subset.
- **`and`/`or`/`not` on boolean Series** — Python's boolean operators don't vectorize; use
  `&`/`|`/`~` with parentheses around each condition (§6).
- **`.apply(axis=1)` as a first instinct** — almost always slower than a vectorized expression or
  a groupby method (§16); reach for it only when no vectorized equivalent exists.
- **Silent dtype upcasting on missing data** — an integer column with any `NaN` becomes `float64`
  (`NaN` has no integer representation); use pandas' nullable `Int64` dtype if you need integers
  with gaps.
- **Comparing floats with `==`** — same rule as NumPy's notebook §18: use `np.isclose`/rounding,
  not exact equality, for computed float columns.
- **Forgetting `ignore_index=True` on `concat`** — without it, the row index from each input
  DataFrame is kept as-is, which can silently produce duplicate index labels.
